In [8]:
# import libraries
import pandas as pd
import numpy as np
from rich.console import Console
from rich.table import Table
from rich.box import ROUNDED

In [3]:
# load the dataset
data = pd.read_csv('data_cleaned.csv')

In [5]:
pd.set_option('display.max_columns', None) 
print("data sample:")
data.sample(5)

data sample:


,customer_id,age,gender,city,education_level,occupation,stress_level,anxiety_score,self_esteem,impulsiveness,optimism_score,life_satisfaction,social_media_dependency,total_orders,total_spent,campaign_exposure,discount_received,loyalty_points,monthly_purchase_amount,weekly_site_visits,average_order_value
264820,364872,26,Male,Mashhad,Master,Teacher,4.09,4.43,4.83,7.20,7.48,6.30,4.91,9,789.02,5,4.87,86.150,102.96,5.500992,2.952419
207710,307752,36,Female,Tabriz,Bachelor,Teacher,2.93,1.57,4.85,7.27,9.73,6.78,8.73,11,833.63,9,14.17,58.960,169.76,5.652707,2.893250
519799,619888,37,Female,Mashhad,High School,Freelancer,4.71,3.06,5.74,6.05,7.59,7.48,6.75,17,2314.35,11,20.22,306.905,172.55,6.198692,3.121504
386820,486898,42,Male,Mashhad,High School,Manager,8.97,7.64,1.55,5.19,6.66,5.01,6.06,21,2673.40,9,24.79,306.905,225.66,5.867934,3.155424
38662,138672,25,Female,Mashhad,Master,Engineer,4.56,5.72,3.53,6.37,5.95,4.88,3.12,5,308.17,6,9.94,36.070,81.82,4.279267,2.806550


برسی کلی افراد دیتاست 

In [28]:
# Friendly labels for each statistic
_STAT_LABELS = {
    "count": "Count",
    "min": "Minimum",
    "max": "Maximum",
    "mean": "Mean",
    "median": "Median"
}
 
 
def show_descriptive_stats(
    data: pd.DataFrame,
    column: str,
    title: str | None = None,
) -> None:
    """
    Compute and print descriptive statistics for a numeric column as a
    nicely formatted rich table.
 
    Args:
        data: DataFrame containing the column.
        column: Name of the numeric column to summarize.
        title: Optional custom title (defaults to "<Column> Descriptive Statistics").
    """
    stats = data[column].agg(["count", "min", "max", "mean", "median"])
 
    table = Table(
        title=title or f"{column.capitalize()} Descriptive Statistics",
        box=ROUNDED,
        title_style="#A784B6",
        header_style="#842958",
        border_style="dim",
    )
    table.add_column("Statistic", style="bold")
    table.add_column("Value", justify="right", style="bright_white")
 
    for stat, value in stats.items():
        if pd.isna(value):
            value_str = "N/A"
        elif stat == "count":
            value_str = f"{int(value):,}"
        else:
            value_str = f"{value:,.2f}"
        table.add_row(_STAT_LABELS.get(stat, stat.capitalize()), value_str)
 
    Console().print(table)

In [29]:
show_descriptive_stats(data, column="age")

    Age Descriptive    
      Statistics       
╭───────────┬─────────╮
│ Statistic │   Value │
├───────────┼─────────┤
│ Count     │ 549,910 │
│ Minimum   │   18.00 │
│ Maximum   │   75.00 │
│ Mean      │   35.29 │
│ Median    │   35.00 │
╰───────────┴─────────╯

In [30]:
# feature engineering: create age groups
bins = [18, 25, 30, 40, 50, 60, 70, 80]
labels = ['18-25', '26-30', '31-40', '41-50', '51-60', '61-70', '71-80']
data['Age_Group'] = pd.cut(data['age'], bins=bins, labels=labels, right=False)

In [38]:
def show_value_counts(
    data: pd.DataFrame,
    column: str,
    title: str | None = None,
) -> None:
    """
    Display sorted value counts for a categorical column as a styled rich
    table, including each group's percentage share and a total row.
 
    Args:
        data: DataFrame containing the column.
        column: Name of the categorical column to summarize.
        title: Optional custom title (defaults to "<Column> Distribution").
    """
    counts = data[column].value_counts().sort_index()
    total = counts.sum()
 
    display_name = column.replace("_", " ")
    table = Table(
        title=title or f"{display_name} Distribution",
        box=ROUNDED,
        title_style="#A784B6",
        header_style="#842958",
        border_style="dim",
    )
    table.add_column(display_name, style="bold")
    table.add_column("Count", justify="right", style="bright_white")
    table.add_column("Percent", justify="right", style="bright_white")
 
    for group, count in counts.items():
        table.add_row(str(group), f"{count:,}", f"{count / total:.1%}")
 
    table.add_section()
     
 
    Console().print(table)

In [42]:
# value counts for the Age_Group column
show_value_counts(data, column="Age_Group")

     Age Group Distribution      
╭───────────┬─────────┬─────────╮
│ Age Group │   Count │ Percent │
├───────────┼─────────┼─────────┤
│ 18-25     │  93,573 │   17.0% │
│ 26-30     │  76,065 │   13.8% │
│ 31-40     │ 192,845 │   35.1% │
│ 41-50     │ 135,910 │   24.7% │
│ 51-60     │  44,314 │    8.1% │
│ 61-70     │   6,719 │    1.2% │
│ 71-80     │     484 │    0.1% │
╰───────────┴─────────┴─────────╯

In [44]:
show_value_counts(data, column="gender")

     gender Distribution      
╭────────┬─────────┬─────────╮
│ gender │   Count │ Percent │
├────────┼─────────┼─────────┤
│ Female │ 286,109 │   52.0% │
│ Male   │ 263,801 │   48.0% │
╰────────┴─────────┴─────────╯

In [ ]:
show_value_counts(data, column="city")

      city Distribution       
╭─────────┬────────┬─────────╮
│ city    │  Count │ Percent │
├─────────┼────────┼─────────┤
│ Isfahan │ 91,921 │   16.7% │
│ Karaj   │ 91,606 │   16.7% │
│ Mashhad │ 91,841 │   16.7% │
│ Shiraz  │ 91,636 │   16.7% │
│ Tabriz  │ 91,893 │   16.7% │
│ Tehran  │ 91,013 │   16.6% │
╰─────────┴────────┴─────────╯

In [46]:
show_value_counts(data, column="education_level")

     education level Distribution      
╭─────────────────┬─────────┬─────────╮
│ education level │   Count │ Percent │
├─────────────────┼─────────┼─────────┤
│ Bachelor        │ 247,077 │   44.9% │
│ High School     │ 137,955 │   25.1% │
│ Master          │ 126,393 │   23.0% │
│ PhD             │  38,485 │    7.0% │
╰─────────────────┴─────────┴─────────╯

In [ ]:
show_value_counts(data, column="occupation")

     occupation Distribution     
╭────────────┬────────┬─────────╮
│ occupation │  Count │ Percent │
├────────────┼────────┼─────────┤
│ Analyst    │ 89,087 │   16.2% │
│ Engineer   │ 88,639 │   16.1% │
│ Freelancer │ 89,094 │   16.2% │
│ Manager    │ 88,873 │   16.2% │
│ Student    │ 88,815 │   16.2% │
│ Teacher    │ 88,967 │   16.2% │
│ Unknown    │ 16,435 │    3.0% │
╰────────────┴────────┴─────────╯